## **EDA - Detección de Problemas de Calidad**


In [0]:
%sql

-- VALORES INVALIDOS EN PRECIOS
WITH tickets AS (

  SELECT
    COUNT(*) AS ticket_total
  FROM ventas_clean 
),
problemas_calidad AS (

  SELECT
    COUNT(CASE WHEN precio IS NULL OR precio <= 0.01 THEN 1 END ) AS precio_invalido,
    COUNT(CASE WHEN cant IS NULL OR cant <= 0 THEN 1 END)  AS cant_invalido,
    COUNT(CASE WHEN total IS NULL OR total <= 0.01 THEN 1 END)  AS total_invalido
  FROM ventas_clean
)

SELECT
  t.ticket_total,
  p.precio_invalido,
  ROUND(p.precio_invalido * 100.0 / t.ticket_total ,2) AS pct_precio_invalido,
  
  ROUND(p.cant_invalido * 100.0 / t.ticket_total,2) AS pct_cant_invalido,

  p.total_invalido,
  ROUND(p.total_invalido * 100.0 / t.ticket_total,2) AS pct_total_invalido
FROM tickets t, problemas_calidad p


In [0]:
%sql

-- REGISTROS DUPLICADOS
WITH duplicados AS (
  SELECT 
    venta, 
    articulo, 
    descrip,
    cant,
    ROW_NUMBER() OVER (
      PARTITION BY venta, articulo, descrip,cant
      ORDER BY venta
    ) AS rn
  FROM ventas_clean
)
SELECT * FROM duplicados WHERE rn > 1;

In [0]:
%sql
--MISMO ARTICULO CON DIFERENTAS DESCRIPCIONES
SELECT
  articulo,
  COUNT(DISTINCT TRIM(LOWER(descrip))) AS distintas_descripciones
FROM ventas_clean
GROUP BY articulo 
  HAVING COUNT(DISTINCT TRIM(LOWER(descrip))) > 1
  ORDER BY distintas_descripciones DESC

In [0]:
%sql

-- OUTLIER EN PRECIO POR ARTICULO
WITH outliers AS (
  SELECT
    articulo,
    ROUND(PERCENTILE(precio,0.01),2)AS p1,
    ROUND(PERCENTILE(precio,0.99),2) AS p99
  FROM ventas_clean
    WHERE precio > 0 
    GROUP BY articulo
)

SELECT 
  v.articulo,
  'OUTLIER_BAJO' AS tipo_outlier,
  COUNT(*) AS total
FROM ventas_clean v
JOIN outliers o
  ON v.articulo = o.articulo
  WHERE v.precio < o.p1
  GROUP BY v.articulo

UNION ALL

SELECT 
  v.articulo,
  'OUTLIER_ALTO' AS tipo_outlier,
  COUNT(*) AS total
FROM ventas_clean v
JOIN outliers o
  ON v.articulo = o.articulo
  WHERE v.precio > o.p99
  GROUP BY v.articulo